In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


import os
import sys
sys.path.append(os.path.abspath(".."))
from src.utils.paths import RAW_DATA_DIR, DATA_DIR
from src.utils.month_codes import MONTH_CODE

import warnings
warnings.filterwarnings('ignore')

In [2]:
agent_path=DATA_DIR/'agent_right'
futures_path=DATA_DIR/'futures_right'

market='Nasdaq'
agent_data=pd.read_parquet(agent_path/f'{market}.parquet')
futures_5=pd.read_parquet(futures_path/'minute_5'/f'{market}.parquet')
futures_prep_5=pd.read_parquet(futures_path/'minute_5'/f'{market}_prep.parquet')
futures_1=pd.read_parquet(futures_path/'minute_1'/f'{market}.parquet')
futures_prep_1=pd.read_parquet(futures_path/'minute_1'/f'{market}_prep.parquet')

print(agent_data.shape)
print(futures_5.shape)
print(futures_prep_5.shape)
print(futures_1.shape)
print(futures_prep_1.shape)

(29532, 15)
(29532, 9)
(3316, 8)
(147660, 10)
(16506, 8)


agent data has 6 useful columns: [date, hour, minute, price, position, time]. data, hour, minute, time are easy to understand, but all of them are "trading time", which equals clock time + 6 hours.

price is 5-minute close price. For example, price at 2020-01-02 06:00:00 is 8787.75, which is the close price of the time interval [5:55,6:00); price at 2020-01-02 06:10:00 is 8786.00, which is the close price of the interval [6:05,6:10).

position means net position. If net position changes, there is a trade. For example, at 2020-01-02 08:30:00, net position changed from 0 to 3. It means the agent bought 3 units. But at what price? Professor said using the immediate price, which is the price at 2020-01-02 08:30:00, 8795.00, which is the close price of the interval [8:25,8:30). Note the position's time and price's time is same.

So, our assumption is that: at 2020-01-02 08:30:00, the agent received a buying demand, and he immediately executed at the market price. What we want to do is that: we don't need to execute immediately, we can time the market. In this case, we have a buying demand. If we predict the market has an upward trend, we buy immediately as the agent did; but if we predict the market has a downward trend, we don't want to buy, we can hold this demand and wait until the market enters another bullish period.

This seems to be quite different from the description in the project PDF file, where professer provided a method to caculate filling probability conditional on volatility/momentum regime. I think we can take two steps:

step 1: just use techincal indicators to predict the market trend. For example, combine short/long horizon exponential moving average of price, if short-term average is larger than long-term average, we predict there will be an upward trend, and we execute all buying orders. This is a "macro-level" step, we use low-frequency data (10,20,30..-minute) to predict, we don't care about the microstructure/filling probability.

step 2: based on the timing in step 1, when we decide to execute buying orders based on upward prediction, we can look at the filling probability now. Set a time window (how the frequency is high, the time length is low, maybe 5,10 minutes), set a target price, if filled, then earn money, if not, then execute at the close price of that time window.

In [3]:
agent_data.head()

,date,hour,minute,price,position,time,matched_contracts,matched_contract_ranks,matched_contract_volumes,match_status,chosen_contract,chosen_contract_rank,chosen_contract_volume,front_rank,front_contract
0,2020-01-02,6,0,8787.75,0,2020-01-02 06:00:00,[NQH20],[1],[77],done,NQH20,1.0,77.0,1,NQH20
1,2020-01-02,6,5,8787.75,0,2020-01-02 06:05:00,[NQH20],[1],[98],done,NQH20,1.0,98.0,1,NQH20
2,2020-01-02,6,10,8786.00,0,2020-01-02 06:10:00,[NQH20],[1],[120],done,NQH20,1.0,120.0,1,NQH20
3,2020-01-02,6,15,8785.50,0,2020-01-02 06:15:00,[NQH20],[1],[170],done,NQH20,1.0,170.0,1,NQH20
4,2020-01-02,6,20,8785.25,0,2020-01-02 06:20:00,[NQH20],[1],[154],done,NQH20,1.0,154.0,1,NQH20


In [4]:
agent_data.loc[28:32]

,date,hour,minute,price,position,time,matched_contracts,matched_contract_ranks,matched_contract_volumes,match_status,chosen_contract,chosen_contract_rank,chosen_contract_volume,front_rank,front_contract
28,2020-01-02,8,20,8794.50,0,2020-01-02 08:20:00,[NQH20],[1],[738],done,NQH20,1.0,738.0,1,NQH20
29,2020-01-02,8,25,8795.25,0,2020-01-02 08:25:00,[NQH20],[1],[546],done,NQH20,1.0,546.0,1,NQH20
30,2020-01-02,8,30,8795.00,3,2020-01-02 08:30:00,[NQH20],[1],[181],done,NQH20,1.0,181.0,1,NQH20
31,2020-01-02,8,35,8794.75,3,2020-01-02 08:35:00,[NQH20],[1],[265],done,NQH20,1.0,265.0,1,NQH20
32,2020-01-02,8,40,8795.50,3,2020-01-02 08:40:00,[NQH20],[1],[251],done,NQH20,1.0,251.0,1,NQH20


futures_5 is 5-minute interval data. It has the same timestamp as the agent data, so it's easy to use merge or other data processing functions, it's also easy to match the orders.

The useful columns are [time, open, low, close, volume]

here close is the same price as price in the agent data. They are matched. Other prices and volumes use the same time interval as close.

In [5]:
futures_5.head(10)

,time,contract,rank,match_status,open,high,low,close,volume
0,2020-01-02 06:00:00,NQH20,1,done,8787.25,8788.00,8787.25,8787.75,77.0
1,2020-01-02 06:05:00,NQH20,1,done,8787.75,8788.25,8787.25,8787.75,98.0
2,2020-01-02 06:10:00,NQH20,1,done,8787.75,8787.75,8786.00,8786.00,120.0
3,2020-01-02 06:15:00,NQH20,1,done,8786.00,8786.75,8785.50,8785.50,170.0
4,2020-01-02 06:20:00,NQH20,1,done,8785.50,8786.75,8784.50,8785.25,154.0
5,2020-01-02 06:25:00,NQH20,1,done,8785.25,8785.25,8784.50,8784.50,101.0
6,2020-01-02 06:30:00,NQH20,1,done,8784.50,8784.75,8784.00,8784.00,162.0
7,2020-01-02 06:35:00,NQH20,1,done,8784.00,8784.75,8783.50,8784.00,159.0
8,2020-01-02 06:40:00,NQH20,1,done,8784.25,8786.25,8784.00,8786.25,117.0
9,2020-01-02 06:45:00,NQH20,1,done,8786.00,8788.00,8786.00,8787.50,166.0


future_prep_5 is also 5-minute interval data. prep means preperation. It cantains data before the first timestamp of the agent data.

useful columsn are [time, open, high, low, close, volume]

The meaning of these columns are the same as those of future_5's columns.

I forgot to reset the index when cleaning the data, you can reset it.

In [6]:
futures_prep_5.tail(10)

,time,open,high,low,close,volume,contract,rank
3306,2020-01-02 05:10:00,8790.50,8790.50,8790.00,8790.50,69,NQH20,1
3307,2020-01-02 05:15:00,8790.50,8790.50,8788.75,8789.25,150,NQH20,1
3308,2020-01-02 05:20:00,8789.50,8789.50,8788.50,8789.00,61,NQH20,1
3309,2020-01-02 05:25:00,8789.00,8789.00,8788.25,8788.50,94,NQH20,1
3310,2020-01-02 05:30:00,8788.50,8789.00,8788.25,8788.75,63,NQH20,1
3311,2020-01-02 05:35:00,8789.00,8789.75,8788.75,8789.75,58,NQH20,1
3312,2020-01-02 05:40:00,8789.75,8790.00,8789.50,8789.75,43,NQH20,1
3313,2020-01-02 05:45:00,8789.75,8789.75,8788.50,8789.50,63,NQH20,1
3314,2020-01-02 05:50:00,8789.25,8789.25,8788.50,8789.00,35,NQH20,1
3315,2020-01-02 05:55:00,8788.75,8789.25,8787.00,8787.50,218,NQH20,1


futures_1 is 1_minute interval data. If you want to calculate high-frequency features, you can use this.

the useful collumsn are [time, open, high, low, close, volume]

notice the meaning of time here. for example, 2020-01-02 05:55:00 refers to [5:55, 5:56); 2020-01-02 06:00:00 refers to [6:00, 6:01).

the agent_time column is also worth noticing. For example, 2020-01-02 05:55:00 -- 2020-01-02 05:59:00 all have the same agent_time: 2020-01-02 06:00:00, it means the price in the agent data is calculated using these 5 rows. These five rows contain 1-minute information in the [5:55, 6:00) time interval.

In [7]:
futures_1.head(10)

,time,open,high,low,close,volume,contract,rank,agent_time,match_status
0,2020-01-02 05:55:00,8787.25,8787.75,8787.25,8787.25,22.0,NQH20,1,2020-01-02 06:00:00,done
1,2020-01-02 05:56:00,8787.50,8788.00,8787.25,8788.00,30.0,NQH20,1,2020-01-02 06:00:00,done
2,2020-01-02 05:57:00,8787.75,8788.00,8787.75,8788.00,6.0,NQH20,1,2020-01-02 06:00:00,done
3,2020-01-02 05:58:00,8788.00,8788.00,8787.75,8787.75,13.0,NQH20,1,2020-01-02 06:00:00,done
4,2020-01-02 05:59:00,8787.50,8787.75,8787.50,8787.75,6.0,NQH20,1,2020-01-02 06:00:00,done
5,2020-01-02 06:00:00,8787.75,8788.25,8787.75,8788.25,28.0,NQH20,1,2020-01-02 06:05:00,done
6,2020-01-02 06:01:00,8788.25,8788.25,8787.25,8787.25,37.0,NQH20,1,2020-01-02 06:05:00,done
7,2020-01-02 06:02:00,8787.50,8787.75,8787.50,8787.50,5.0,NQH20,1,2020-01-02 06:05:00,done
8,2020-01-02 06:03:00,8787.25,8787.75,8787.25,8787.50,10.0,NQH20,1,2020-01-02 06:05:00,done
9,2020-01-02 06:04:00,8787.50,8787.75,8787.50,8787.75,18.0,NQH20,1,2020-01-02 06:05:00,done


It has the same retionship to futures_1 as futures_prep_5 has to futures_5.

Also, I forgot to reset the index, you can reset it.

In [8]:
futures_prep_1.head()

,time,open,high,low,close,volume,contract,rank
0,2019-12-13 00:00:00,8508.00,8521.50,8508.00,8517.50,297,NQH20,1
1,2019-12-13 00:01:00,8517.50,8518.50,8516.25,8518.00,116,NQH20,1
2,2019-12-13 00:02:00,8518.25,8518.25,8515.75,8516.75,102,NQH20,1
3,2019-12-13 00:03:00,8516.75,8519.25,8516.75,8517.75,65,NQH20,1
4,2019-12-13 00:04:00,8518.75,8518.75,8517.50,8518.00,45,NQH20,1
